# 0 · Import Contracts to Volume
Copies the **fictional sample contract PDFs** from the `contracts/` folder in this
repo into a Unity Catalog **Volume**. That volume is the knowledge source for the
Vector Search index in notebook `1`.
**What this notebook does**
1. Creates the target catalog / schema / `contracts` volume (if missing)
2. Copies every `*.pdf` from `CONTRACTS_SOURCE_PATH` into the volume
3. Verifies the file count
> Import this repo folder to your workspace so `./contracts` sits next to this
> notebook (see `3_agent-bricks/contracts/` in GitHub).


### Define Parameters
| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `marketing_campaign_analytics` | Unity Catalog **catalog** for all demo objects (tables, volumes, indexes). |
| `SCHEMA_NAME` | `marketing_campaign` | Schema inside the catalog for sales data and contracts. |
| `VOLUME_NAME` | `contracts` | Volume that stores the contract PDFs for parsing. |
| `CONTRACTS_SOURCE_PATH` | `./contracts` | Folder **next to this notebook** containing the PDFs (no zip required). |


In [ ]:
dbutils.widgets.text("CATALOG_NAME", "marketing_campaign_analytics", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("VOLUME_NAME", "contracts", "VOLUME_NAME — UC volume for contract PDFs")
dbutils.widgets.text("CONTRACTS_SOURCE_PATH", "./contracts", "CONTRACTS_SOURCE_PATH — local folder with PDFs")
CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
CONTRACTS_SOURCE_PATH = dbutils.widgets.get("CONTRACTS_SOURCE_PATH")
TARGET_DIR = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
print("Contracts volume :", TARGET_DIR)
print("Source folder    :", CONTRACTS_SOURCE_PATH)


### Create Catalog, Schema, and Volume


In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")
print("Catalog, schema, and volume are ready.")


### Copy PDFs from the Repo Folder into the Volume
Each file in `contracts/` is copied flat into the volume. The demo ships four
fictional contracts (`fictional_*.pdf`).


In [ ]:
import os
import shutil
assert os.path.isdir(CONTRACTS_SOURCE_PATH), (
    f"Folder not found: {CONTRACTS_SOURCE_PATH}. Import the repo so ./contracts sits next to this notebook."
)
copied = []
for fn in sorted(os.listdir(CONTRACTS_SOURCE_PATH)):
    if not fn.lower().endswith(".pdf"):
        continue
    src = os.path.join(CONTRACTS_SOURCE_PATH, fn)
    dst = os.path.join(TARGET_DIR, fn)
    shutil.copyfile(src, dst)
    copied.append(fn)
assert copied, f"No PDF files found in {CONTRACTS_SOURCE_PATH}"
print(f"Copied {len(copied)} PDFs into {TARGET_DIR}")
for fn in copied:
    print("  ", fn)


### Verify


In [ ]:
files = [f for f in os.listdir(TARGET_DIR) if f.lower().endswith(".pdf")]
print(f"{len(files)} PDF files now in {TARGET_DIR}")
